In [2]:

# ============================================================
# PROJECT 9: PREDICTIVE MAINTENANCE INTELLIGENCE AGENT
# Synthetic manufacturing data | Interactive Gradio dashboard
# ============================================================

!pip -q install gradio plotly pandas numpy

import pandas as pd
import numpy as np
import gradio as gr
import plotly.express as px
import plotly.graph_objects as go

# -------------------- 1. SYNTHETIC DATA --------------------

np.random.seed(42)

def generate_data(n=300):
    machines = [f"M-{i:03d}" for i in range(1, 21)]
    departments = ["Assembly", "Packaging", "Machining", "Quality Control"]
    machine_types = ["Motor", "Conveyor", "Pump", "Compressor"]

    df = pd.DataFrame({
        "Machine_ID": np.random.choice(machines, n),
        "Department": np.random.choice(departments, n),
        "Machine_Type": np.random.choice(machine_types, n),
        "Operating_Hours": np.random.randint(100, 9001, n),
        "Temperature_C": np.round(np.random.normal(72, 13, n).clip(35, 125), 1),
        "Vibration_mm_s": np.round(np.random.normal(4.2, 2.0, n).clip(0.3, 13), 2),
        "Pressure_Bar": np.round(np.random.normal(6, 1.2, n).clip(2, 11), 2),
        "Days_Since_Maintenance": np.random.randint(1, 181, n),
        "Previous_Failures": np.random.randint(0, 6, n),
        "Scheduled_Production_Hours": np.random.randint(4, 25, n),
        "Maintenance_Cost": np.random.randint(500, 12001, n),
        "Downtime_Hours": np.round(np.random.exponential(2.2, n).clip(0, 24), 1)
    })

    return df

# -------------------- 2. RISK ENGINE --------------------

def analyze_machines(data):
    df = data.copy()

    required = [
        "Machine_ID", "Temperature_C", "Vibration_mm_s",
        "Pressure_Bar", "Operating_Hours",
        "Days_Since_Maintenance", "Previous_Failures",
        "Scheduled_Production_Hours"
    ]

    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError("Missing required columns: " + ", ".join(missing))

    numeric = [
        "Temperature_C", "Vibration_mm_s", "Pressure_Bar",
        "Operating_Hours", "Days_Since_Maintenance",
        "Previous_Failures", "Scheduled_Production_Hours"
    ]

    for c in numeric:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.dropna(subset=required).copy()
    if df.empty:
        raise ValueError("No valid records remain after data validation.")

    defaults = {
        "Department": "Unspecified",
        "Machine_Type": "Unspecified",
        "Maintenance_Cost": 0,
        "Downtime_Hours": 0
    }

    for col, default in defaults.items():
        if col not in df.columns:
            df[col] = default

    for col in ["Maintenance_Cost", "Downtime_Hours"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    # Transparent illustrative risk scoring rules
    df["Temperature_Risk"] = np.select(
        [df["Temperature_C"] >= 100, df["Temperature_C"] >= 85],
        [30, 18], default=0
    )

    df["Vibration_Risk"] = np.select(
        [df["Vibration_mm_s"] >= 8, df["Vibration_mm_s"] >= 6],
        [25, 15], default=0
    )

    df["Pressure_Risk"] = np.select(
        [(df["Pressure_Bar"] < 3) | (df["Pressure_Bar"] > 9),
         (df["Pressure_Bar"] < 4) | (df["Pressure_Bar"] > 8)],
        [20, 10], default=0
    )

    df["Maintenance_Risk"] = np.select(
        [df["Days_Since_Maintenance"] >= 120,
         df["Days_Since_Maintenance"] >= 75],
        [15, 8], default=0
    )

    df["Failure_History_Risk"] = np.select(
        [df["Previous_Failures"] >= 4,
         df["Previous_Failures"] >= 2],
        [15, 8], default=0
    )

    df["Operating_Risk"] = np.where(
        df["Operating_Hours"] >= 7000, 10, 0
    )

    risk_cols = [
        "Temperature_Risk", "Vibration_Risk", "Pressure_Risk",
        "Maintenance_Risk", "Failure_History_Risk", "Operating_Risk"
    ]

    df["Risk_Score"] = df[risk_cols].sum(axis=1).clip(0, 100)

    df["Risk_Level"] = pd.cut(
        df["Risk_Score"],
        bins=[-1, 29, 59, 100],
        labels=["Low", "Medium", "High"]
    ).astype(str)

    def recommend(row):
        actions = []

        if row["Temperature_C"] >= 85:
            actions.append("Inspect overheating; check cooling and lubrication")
        if row["Vibration_mm_s"] >= 6:
            actions.append("Inspect bearings, alignment, and vibration")
        if row["Pressure_Bar"] < 4 or row["Pressure_Bar"] > 8:
            actions.append("Inspect pressure system and possible leaks")
        if row["Days_Since_Maintenance"] >= 75:
            actions.append("Review preventive maintenance schedule")
        if row["Previous_Failures"] >= 2:
            actions.append("Review failure history and root causes")
        if row["Operating_Hours"] >= 7000:
            actions.append("Assess wear and plan inspection")

        if not actions:
            actions.append("Continue routine monitoring and scheduled servicing")

        if row["Risk_Level"] == "High":
            urgency = "Priority 1 — inspect promptly; assess safe operation"
        elif row["Risk_Level"] == "Medium":
            urgency = "Priority 2 — schedule inspection and monitor sensors"
        else:
            urgency = "Priority 3 — continue routine preventive maintenance"

        return urgency + ". " + "; ".join(actions) + "."

    df["Recommendation"] = df.apply(recommend, axis=1)
    return df

# -------------------- 3. CSV UPLOAD SUPPORT --------------------

ALIASES = {
    "machine_id": "Machine_ID",
    "machine": "Machine_ID",
    "temperature": "Temperature_C",
    "temperature_c": "Temperature_C",
    "vibration": "Vibration_mm_s",
    "vibration_mm_s": "Vibration_mm_s",
    "pressure": "Pressure_Bar",
    "pressure_bar": "Pressure_Bar",
    "operating_hours": "Operating_Hours",
    "days_since_maintenance": "Days_Since_Maintenance",
    "previous_failures": "Previous_Failures",
    "scheduled_production_hours": "Scheduled_Production_Hours",
    "department": "Department",
    "machine_type": "Machine_Type",
    "maintenance_cost": "Maintenance_Cost",
    "downtime_hours": "Downtime_Hours"
}

def load_data(file):
    if file is None:
        return generate_data()

    df = pd.read_csv(file.name if hasattr(file, "name") else file)
    df.columns = [
        ALIASES.get(str(c).strip().lower().replace(" ", "_"), c)
        for c in df.columns
    ]
    return df

# -------------------- 4. DASHBOARD --------------------

def dashboard(file, department, risk_level, machine_type):
    try:
        raw = load_data(file)
        df = analyze_machines(raw)

        if department != "All":
            df = df[df["Department"].astype(str) == department]
        if risk_level != "All":
            df = df[df["Risk_Level"] == risk_level]
        if machine_type != "All":
            df = df[df["Machine_Type"].astype(str) == machine_type]

        if df.empty:
            return (
                "No machines match the selected filters.",
                None, None, None, None, None, None
            )

        high = int((df["Risk_Level"] == "High").sum())
        medium = int((df["Risk_Level"] == "Medium").sum())
        downtime = round(df["Downtime_Hours"].sum(), 1)
        avg_risk = round(df["Risk_Score"].mean(), 1)

        summary = f"""
## Predictive Maintenance Control Centre

**Machines analyzed:** {len(df)}
**High-risk records:** {high}
**Medium-risk records:** {medium}
**Total recorded downtime:** {downtime} hours
**Average risk score:** {avg_risk}/100

**Recommended focus:** Review high-risk records first, validate sensor readings,
and coordinate inspections around production schedules.
"""

        risk_fig = px.histogram(
            df, x="Risk_Level", color="Risk_Level",
            category_orders={"Risk_Level": ["High", "Medium", "Low"]},
            title="Machine Risk Distribution",
            labels={"Risk_Level": "Risk Category", "count": "Records"}
        )

        scatter_fig = px.scatter(
            df, x="Temperature_C", y="Vibration_mm_s",
            color="Risk_Level", size="Previous_Failures",
            hover_data=["Machine_ID", "Department", "Risk_Score"],
            title="Sensor Health: Temperature vs Vibration"
        )

        trend_fig = px.scatter(
            df, x="Operating_Hours", y="Risk_Score",
            color="Risk_Level", hover_data=["Machine_ID"],
            title="Operating Hours vs Maintenance Risk"
        )

        dept_fig = px.bar(
            df.groupby("Department", as_index=False)["Downtime_Hours"].sum(),
            x="Department", y="Downtime_Hours",
            title="Recorded Downtime by Department"
        )

        priority = df.sort_values(
            ["Risk_Score", "Downtime_Hours"], ascending=False
        )

        alert_cols = [
            "Machine_ID", "Department", "Machine_Type",
            "Risk_Score", "Risk_Level", "Recommendation"
        ]

        detail_cols = [
            "Machine_ID", "Department", "Machine_Type",
            "Temperature_C", "Vibration_mm_s", "Pressure_Bar",
            "Operating_Hours", "Days_Since_Maintenance",
            "Previous_Failures", "Downtime_Hours",
            "Risk_Score", "Risk_Level", "Recommendation"
        ]

        alerts = priority[alert_cols].head(15)
        details = priority[[c for c in detail_cols if c in priority.columns]]

        methodology = """
### How the agent works
The prototype assigns points for unusual temperature, vibration, pressure,
overdue maintenance, previous failures, and high operating hours. Scores are
capped at 100. Scores 0–29 are Low, 30–59 Medium, and 60–100 High.

These are illustrative rules, not validated failure probabilities. Sensor limits
must be adjusted to each machine's specifications. The agent supports maintenance
decision-making and does not replace safety procedures or engineering inspections.
"""

        return summary, risk_fig, scatter_fig, trend_fig, dept_fig, alerts, details

    except Exception as e:
        return (
            f"**Data error:** {str(e)}",
            None, None, None, None, pd.DataFrame(), pd.DataFrame()
        )


# ============================================================
# 5. PREMIUM INDUSTRIAL CONTROL-ROOM INTERFACE
# Requires Sections 1-4 from the existing notebook
# ============================================================

custom_css = """
.gradio-container {
    max-width: 1500px !important;
    margin: auto !important;
    background: #0b1120 !important;
    color: #edf3ff !important;
    font-family: Inter, Arial, sans-serif !important;
}
body { background: #0b1120 !important; }

.hero {
    background: linear-gradient(120deg, #111d36 0%, #172d50 58%, #123d49 100%);
    border: 1px solid #294769;
    border-radius: 20px;
    padding: 30px;
    margin-bottom: 18px;
}
.hero h1 {
    font-size: 32px !important;
    color: #f5f8ff !important;
    margin: 8px 0 !important;
}
.hero p { color: #b8c9e5; font-size: 14px; }
.eyebrow {
    color: #5eead4;
    letter-spacing: 2px;
    font-weight: 700;
    font-size: 11px;
}
.section-title {
    color: #eaf2ff;
    font-size: 19px;
    font-weight: 700;
    padding: 12px 0 5px 0;
}
.status-pill {
    display: inline-block;
    background: #123e3d;
    color: #6ee7d8;
    padding: 7px 12px;
    border-radius: 20px;
    font-size: 12px;
    border: 1px solid #23645e;
}
.kpi-wrap {
    display: grid;
    grid-template-columns: repeat(4, minmax(0, 1fr));
    gap: 12px;
    margin: 14px 0;
}
.kpi {
    background: linear-gradient(145deg, #17233a, #101a2d);
    border: 1px solid #293a56;
    border-radius: 15px;
    padding: 17px;
    min-height: 92px;
}
.kpi-label {
    color: #9eb0ca;
    font-size: 12px;
    margin-bottom: 10px;
}
.kpi-value {
    color: #f3f7ff;
    font-size: 27px;
    font-weight: 750;
}
.kpi-note {
    color: #7188a8;
    font-size: 11px;
    margin-top: 6px;
}
.kpi-red { border-top: 3px solid #fb7185; }
.kpi-amber { border-top: 3px solid #fbbf24; }
.kpi-blue { border-top: 3px solid #60a5fa; }
.kpi-teal { border-top: 3px solid #2dd4bf; }

.gr-panel, .panel {
    background: #111b2e !important;
    border: 1px solid #273853 !important;
    border-radius: 15px !important;
}
button.primary {
    background: linear-gradient(90deg, #2563eb, #0d9488) !important;
    color: white !important;
    border: 0 !important;
    border-radius: 10px !important;
    font-weight: 700 !important;
}
label, .prose { color: #dbe7fa !important; }
input, textarea {
    background: #101a2b !important;
    color: #f1f5f9 !important;
    border-color: #344660 !important;
}
footer { display: none !important; }

@media (max-width: 800px) {
    .kpi-wrap { grid-template-columns: repeat(2, minmax(0, 1fr)); }
    .hero h1 { font-size: 24px !important; }
}
"""

initial = generate_data()

with gr.Blocks(
    theme=gr.themes.Base(
        primary_hue="teal",
        secondary_hue="blue",
        neutral_hue="slate"
    ),
    css=custom_css,
    title="Aegis | Predictive Maintenance Intelligence"
) as demo:

    gr.HTML("""
    <div class="hero">
        <div class="eyebrow">AEGIS INDUSTRIAL INTELLIGENCE &nbsp; / &nbsp; OPERATIONS</div>
        <h1>Predictive Maintenance Control Centre</h1>
        <p>Machine health intelligence · Risk prioritization · Preventive action planning</p>
        <span class="status-pill">● SYNTHETIC DATA MODE</span>
    </div>
    """)

    gr.Markdown("### Control filters")
    with gr.Row():
        file = gr.File(
            label="Upload machine CSV",
            file_types=[".csv"],
            type="filepath"
        )
        department = gr.Dropdown(
            ["All"] + sorted(initial["Department"].unique().tolist()),
            value="All",
            label="Department"
        )
        risk = gr.Dropdown(
            ["All", "High", "Medium", "Low"],
            value="All",
            label="Risk priority"
        )
        machine_type = gr.Dropdown(
            ["All"] + sorted(initial["Machine_Type"].unique().tolist()),
            value="All",
            label="Machine type"
        )

    with gr.Row():
        run_btn = gr.Button(
            "RUN MACHINE HEALTH ANALYSIS",
            variant="primary",
            scale=4
        )
        reset_btn = gr.Button("RESET FILTERS", scale=1)

    summary = gr.Markdown(
        elem_classes=["panel"]
    )

    with gr.Tabs():
        with gr.Tab("01  Executive Overview"):
            gr.Markdown("### Risk intelligence")
            with gr.Row():
                risk_plot = gr.Plot(label="Risk distribution")
                dept_plot = gr.Plot(label="Downtime by department")

        with gr.Tab("02  Sensor Intelligence"):
            gr.Markdown("### Machine condition monitoring")
            with gr.Row():
                sensor_plot = gr.Plot(label="Temperature vs vibration")
                trend_plot = gr.Plot(label="Operating hours vs risk")

        with gr.Tab("03  Maintenance Command Queue"):
            gr.Markdown(
                "### Priority worklist\n"
                "Review high-risk records first. Recommendations are decision "
                "support and must be validated by maintenance personnel."
            )
            alert_table = gr.Dataframe(
                label="Priority maintenance alerts",
                wrap=True
            )

        with gr.Tab("04  Machine Records"):
            detail_table = gr.Dataframe(
                label="Filtered machine data and recommendations",
                wrap=True
            )

        with gr.Tab("05  Scoring Method"):
            gr.Markdown("""
            ## How Aegis evaluates machine risk

            | Indicator | Risk points |
            |---|---:|
            | High temperature | Up to 30 |
            | Excessive vibration | Up to 25 |
            | Unusual pressure | Up to 20 |
            | Time since maintenance | Up to 15 |
            | Previous failures | Up to 15 |
            | High operating hours | Up to 10 |

            **Risk bands:** Low = 0–29 · Medium = 30–59 · High = 60–100.

            The original scoring function caps the total at 100. The individual
            points are illustrative and not calibrated failure probabilities.
            Machine-specific limits and maintenance manuals should take
            precedence over these generic thresholds.
            """)

    outputs = [
        summary, risk_plot, sensor_plot, trend_plot,
        dept_plot, alert_table, detail_table
    ]

    def run_analysis(file, department, risk, machine_type):
        return dashboard(file, department, risk, machine_type)

    run_btn.click(
        fn=run_analysis,
        inputs=[file, department, risk, machine_type],
        outputs=outputs
    )

    def reset_dashboard():
        return (
            None, "All", "All", "All",
            *dashboard(None, "All", "All", "All")
        )

    reset_btn.click(
        fn=reset_dashboard,
        inputs=[],
        outputs=[
            file, department, risk, machine_type,
            *outputs
        ]
    )

    demo.load(
        fn=run_analysis,
        inputs=[file, department, risk, machine_type],
        outputs=outputs
    )

demo.launch(share=True)


/tmp/ipykernel_369/3189757982.py:406: UserWarning:

The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b1ae1c579d2404201e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
